# Interpreting the coefficients

The linear regression model for flight duration as a function of distance takes the form  
<img src="Ex2.jpg" align = "left">  


where  

<img src="Ex2_2.jpg" align = "left">

By looking at the coefficients of your model you will be able to infer

- how much of the average flight duration is actually spent on the ground and
- what the average speed is during a flight.
The linear regression model is available as `regression`.

## Instructions

- What's the intercept?
- What are the coefficients? This is a vector.
- Extract the element from the vector which corresponds to the slope for distance.
- Find the average speed in km per hour.

In [4]:
# # Import the SparkSession class
# import pyspark
# from pyspark.sql import SparkSession

# spark = SparkSession.builder.appName('flights').getOrCreate()

In [2]:
# Intialization
import os
import sys

os.environ["SPARK_HOME"] = "/home/talentum/spark"
os.environ["PYLIB"] = os.environ["SPARK_HOME"] + "/python/lib"
# In below two lines, use /usr/bin/python2.7 if you want to use Python 2
os.environ["PYSPARK_PYTHON"] = "/usr/bin/python3.6" 
os.environ["PYSPARK_DRIVER_PYTHON"] = "/usr/bin/python3"
sys.path.insert(0, os.environ["PYLIB"] +"/py4j-0.10.7-src.zip")
sys.path.insert(0, os.environ["PYLIB"] +"/pyspark.zip")

# NOTE: Whichever package you want mention here.
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0 pyspark-shell' 
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'
os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.3 pyspark-shell'
# os.environ['PYSPARK_SUBMIT_ARGS'] = '--packages com.databricks:spark-xml_2.11:0.6.0,org.apache.spark:spark-avro_2.11:2.4.0 pyspark-shell'

In [3]:
#Entrypoint 2.x
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().getOrCreate()

# On yarn:
# spark = SparkSession.builder.appName("Spark SQL basic example").enableHiveSupport().master("yarn").getOrCreate()
# specify .master("yarn")

sc = spark.sparkContext

In [4]:
flights = spark.read.csv('file:///home/talentum/test-jupyter/c5-MLWithPySpark/M3-Regression/2_Regression/dataset/flights.csv',
                         sep=',',
						 header=True,
						 inferSchema=True,
						 nullValue='NA')


from pyspark.sql.functions import round

flights = flights.withColumn('km', round(flights.mile * 1.60934, 0)).drop('mile')

from pyspark.ml.feature import StringIndexer

flights = StringIndexer(inputCol='org', outputCol='org_idx').fit(flights).transform(flights)

# from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import OneHotEncoderEstimator

onehot = OneHotEncoderEstimator(inputCols=['org_idx'], outputCols=['org_dummy'])
#onehot = OneHotEncoder(inputCols=['org_idx'], outputCols=['org_dummy'])

flights = onehot.fit(flights).transform(flights)

from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=['km'], outputCol='features')
flights = assembler.transform(flights)

flights_train, flights_test = flights.randomSplit([0.8, 0.2], seed=13)

from pyspark.ml.regression import LinearRegression
regression = LinearRegression(labelCol='duration').fit(flights_train)


In [ ]:
# Intercept (average minutes on ground)
inter = regression.____
print(inter)

# Coefficients
coefs = ____.____
print(coefs)

# Average minutes per km
minutes_per_km = ____.____[____]
print(minutes_per_km)

# Average speed in km per hour
avg_speed = ____ / ____
print(avg_speed)